In [2]:
import os
import openai
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [22]:
import json

In [ ]:
import pickle

with open(r'movie_id_dict_241016.pkl', 'rb') as file :
    movie_id = pickle.load(file)

with open(r'movie_infos_241016.pkl', 'rb') as file :
    movie_info_1016 = pickle.load(file)


# print(movie_id)
# print(movie_info_1016)

In [11]:
comments = []

movie_name_list = list(movie_info_1016.keys())
for movie_name in movie_name_list :
    comment = movie_info_1016[movie_name]['comments']
    comments.append((movie_name, comment))


# 취향과 감정적 반응 분류
> Q : 질문 없음.

In [5]:
llm = ChatOpenAI( temperature=0.4, 
                 model='gpt-4o-mini')

prompt = """ 너는 사용자의 얘기를 듣고 사용자가 느끼는 감정과 사용자의 취향을 추측하는 심리가야.
사용자의 응답에 듣고 사용자의 원래 취향과 지금 느끼고 있는 감정을 분류해서 json 형태로 반환해. json 형태는 다음과 같아.
사용자의 취향 : 사용자의 원래 취향
사용자의 감정적 상태 : 사용자가 지금 느끼고 있는 감정
단, 사용자의 취향이나 감정적 상태를 신뢰롭게 추측하기 힘들다면 해당 값을 None 값으로 채워.

사용자의 응답 : {query}
JSON object : 
"""


prompt_template = PromptTemplate(template=prompt, input_variables=["comments", "query"])

recom_chain = prompt_template | llm | StrOutputParser()

In [7]:
query = "난 언제 잠을 푹 잘 수 있는걸까? 매일매일 너무 피곤해."
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": None,\n  "사용자의 감정적 상태": "피곤함"\n}\n```'

In [8]:
query = "귀여운건 못참아. 얼마전에 주문했던 귀여운 파츠가 집에 왔어. 너무 귀여워서 신나는거 있지? 아이 기분좋아!"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "귀여운 것",\n  "사용자의 감정적 상태": "신남"\n}\n```'

In [9]:
query = "귀여운건 못참아. 얼마전에 주문했던 귀여운 파츠가 집에 왔어. 파츠를 어디에 붙일지 고민중이야."
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "귀여운 것",\n  "사용자의 감정적 상태": "기대감"\n}\n```'

# MBTI 직접적인 질문
> MBTI 가 뭐야?

In [11]:
llm = ChatOpenAI( temperature=0.4, 
                 model='gpt-4o-mini')

prompt = """ 너는 사용자의 MBTI 를 듣고 사용자의 취향과 감정을 예측하는 MBTI 전문가야.
사용자의 응답에 듣고 사용자의 원래 취향과 지금 느끼고 있는 감정을 분류해서 json 형태로 반환해. json 형태는 다음과 같아.
사용자의 취향 : 사용자의 원래 취향
사용자의 감정적 상태 : 사용자가 지금 느끼고 있는 감정
단, 사용자의 취향이나 감정적 상태를 신뢰롭게 추측하기 힘들다면 해당 값을 None 값으로 채워.

MBTI 에 대한 상세 정보는 다음과 같아.
E : 외향형인 사람들은 다른 사람들과 어울리거나 외부 활동을 통해서 활력을 얻습니다. 일을 할 때에는 혼자서 하는것보다는 다른 사람들과 일할 때 성과가 더 좋은 사람들입니다.
I : 내향형인 사람들은 스스로 에너지를 얻는 걸 선호합니다. 다른 사람들을 통해서가 아니라 자기 내면의 힘으로 동기부여를 받습니다. 혼자 생각할 수 있는 시간을 통해서 활력을 얻는 사람들입니다.
S : 감각형의 사람들은 정보를 접했을 때 그 정보를 감각을 통해 파악합니다. 직접 감각을 통해 느낄 수 있는 정보의 세부사항을 파악하려고 하는 사람들로, 미래를 예측하려고 하기보다는 현재 상황을 중요하게 생각합니다.
N : 직관형 사람들은 정보를 받아들일 때 오감을 이용하는 것 이상으로 직감을 사용합니다. 직관을 통해 미래의 가능성을 판단하고 행동하는 걸 선호하는 사람들입니다.
F : 감정형의 사람들은 어떤 결정을 내릴 때 자신의 감정과 가치관대로 결정하려고 합니다. 다른 사람들과의 관계를 중요하게 생각하며, 다른 사람들의 마음을 잘 이해하는 사람들입니다.
T : 생각형 사람들은 논리적으로 모든 선택지를 꼼꼼하게 결정하려고 합니다. 그렇기 때문에 어떤 결정을 내리는데 많은 시간이 필요합니다. 다른 사람들을 비판하려는 경향이 있습니다.
P : 지각형 사람들은 변화에 적응을 잘합니다. 자발적으로 인생을 사는 사람들로, 계획을 세우는 것보다는 즉흥적이거나 유연하게 행동을 합니다.
J : 판단형 사람들은 정리가 잘 되어있고, 계획을 미리 세워 놓는걸 선호합니다. 일정에 맞게 미리 일처리를 해두어야 마음이 편한 사람들입니다.


사용자의 응답 : {query}
JSON object : 
"""


prompt_template = PromptTemplate(template=prompt, input_variables=["comments"])

recom_chain = prompt_template | llm | StrOutputParser()

In [19]:
query = "ESTJ"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "조직적이고 계획적인 활동을 선호하며, 리더십을 발휘할 수 있는 환경을 좋아함",\n  "사용자의 감정적 상태": null\n}\n```'

In [14]:
query = "ESTJ, 니가 나에 대해 뭘알아"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "조직적이고 계획적인 활동을 선호하며, 리더십을 발휘하는 것을 좋아할 가능성이 높음",\n  "사용자의 감정적 상태": "호기심과 약간의 도전적인 기분"\n}\n```'

In [15]:
query = "ESTJ, 어제 친구를 만났어. 우동도 먹고 맥주도 마셨어. 진짜 맛있었어."
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "음식과 음료를 즐기는 외향적인 활동",\n  "사용자의 감정적 상태": "즐거움"\n}\n```'

In [16]:
query = "ESTJ, 이 세상에 나는 혼자야 울거야"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "조직적이고 계획적인 환경을 선호하며, 타인과의 협력을 중시하는 경향이 있음",\n  "사용자의 감정적 상태": "외로움과 슬픔을 느끼고 있음"\n}\n```'

In [17]:
query = "ESTJ, 로또 당첨되면 좋겠다"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "구조적이고 계획적인 활동을 선호하며, 외부 활동과 사람들과의 상호작용에서 에너지를 얻는 경향이 있음",\n  "사용자의 감정적 상태": "희망적이고 긍정적인 감정"\n}\n```'

In [18]:
query = "ESTJ, 그래도 해야지 어떡해"
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n  "사용자의 취향": "정리정돈을 좋아하고 계획적인 활동을 선호함",\n  "사용자의 감정적 상태": "결단력과 책임감을 느끼고 있음"\n}\n```'

# 사용자에게 필요한 것

In [42]:
llm = ChatOpenAI( temperature=0.7, 
                 model='gpt-4o-mini')

need_prompt = """ 너는 사용자의 얘기를 듣고 사용자가 느끼는 감정과 사용자의 취향을 추측하는 심리가야.
너의 역할은 사용자의 감정과 취향을 분석하여 사용자에게 지금 필요한게 무엇일지 분석하면 돼.
사용자의 응답에 듣고 사용자의 원래 취향과 지금 느끼고 있는 감정을 분류하고 사용자가 필요한 내용을 분석해. json 형태는 다음과 같아.
사용자의 취향 : 사용자의 원래 취향
사용자의 감정적 상태 : 사용자가 지금 느끼고 있는 감정
사용자에게 필요한 것 : 사용자의 원래 취향과 지금의 감정을 고려하여 가장 필요해 보이는 것.
단, 사용자의 취향이나 감정적 상태를 신뢰롭게 추측하기 힘들다면 해당 값을 None 값으로 채워.

사용자의 응답 : {query}
JSON object : 
"""


need_prompt_template = PromptTemplate(template=need_prompt, input_variables=["comments", "query"])

need_chain = need_prompt_template | llm | StrOutputParser()

In [43]:
query1 = "산다는건 뭘까? 요즘 고민이 많이 돼. 왜 모두들 아등바등 살아가고 있는걸까? 조금이라도 쉬면 나만 뒤쳐지는 느낌이 들어."
result1_json = need_chain.invoke({"query":query1})
result1_json

'```json\n{\n  "사용자의 취향": None,\n  "사용자의 감정적 상태": "고민, 불안",\n  "사용자에게 필요한 것": "휴식과 자기 성찰의 시간"\n}\n```'

In [44]:
query2 = "난 요즘 너무 행복해. 인생 뭐 있어? 그냥 되는대로 사는거지! 오늘 하루 숨 쉬며 살아감에 감사해"
result2_json = need_chain.invoke({"query":query2})
result2_json

'```json\n{\n  "사용자의 취향": None,\n  "사용자의 감정적 상태": "행복",\n  "사용자에게 필요한 것": "감사하는 마음을 더욱 키우고 즐거운 순간을 기록할 수 있는 일기나 다이어리를 추천합니다."\n}\n```'

In [45]:
query3 = "요즘 운동을 시작했어. 원래는 유산소를 좋아했는데 근육 펌핑 되는거 보니까 희열이 느껴져. 살아있는 기분 짜릿해"
result3_json = need_chain.invoke({"query":query3})
result3_json

'```json\n{\n  "사용자의 취향": "유산소 운동",\n  "사용자의 감정적 상태": "희열, 짜릿함",\n  "사용자에게 필요한 것": "더 다양한 근력 운동 프로그램이나 동기 부여를 제공하는 커뮤니티"\n}\n```'

In [50]:
query4 = """너무 반짝반짝 눈이 부셔 
No, no, no, no, no
너무 깜짝깜짝 놀란 나는
Oh, oh, oh, oh, oh
너무 짜릿짜릿 몸이 떨려
Gee, gee, gee, gee, gee
Oh, 젖은 눈빛 oh, yeah
Oh, 좋은 향기 oh, yeah, yeah, yeah"""
result4_json = need_chain.invoke({"query":query4})
result4_json

'```json\n{\n  "사용자의 취향": "반짝이는 것, 강렬한 경험",\n  "사용자의 감정적 상태": "놀람, 짜릿함",\n  "사용자에게 필요한 것": "안정감과 편안함을 줄 수 있는 환경"\n}\n```'

# 유용하다고 판단되는 정보

In [48]:
llm = ChatOpenAI( temperature=0.4, 
                 model='gpt-4o-mini')

review_prompt = """ 너는 영화 리뷰 분석 전문가야. 너의 역할은 영화리뷰를 활용해서 각 영화 별로 영화의 취향, 영화의 감정, 그리고 어떤 경험 및 감정을 줄 수 있는 영화인지 분석하면 돼.
영화 리뷰를 보고 영화 별로 어떤 취향의 사람들이 영화를 많이 볼 것 같은지 분석하고, 이 영화를 본 사용자들이 어떤 감정을 느꼈는지 분석해.
또한, 이 영화가 사람들에게 어떤 경험과 감정을 제공할 수 있는 영화인지 분석해. 분석 내용을 json 형태로 반환해. 형태는 다음과 같아.
영화 타겟 취향 : 영화를 본 사용자의 원래 취향
영화 타겟 감정 : 영화를 본 사용자가 느낌 감정
영화 제공 가능 : 영화가 제공할 수 있는 경험과 감정 및 기타 내용
단, 해당 키에 관한 내용을 신뢰롭게 추측하기 힘들다면 해당 값을 None 값으로 채워.

영화 리뷰 : {query}
JSON object : 
"""


review_prompt_template = PromptTemplate(template=review_prompt, input_variables=["query"])

review_chain = review_prompt_template | llm | StrOutputParser()

In [21]:
query = comments
review_json = review_chain.invoke({"query":query})
review_json

'```json\n{\n  "월플라워": {\n    "영화 타겟 취향": "청춘, 성장, 감정적인 이야기 선호",\n    "영화 타겟 감정": "공감, 감동, 질투, 고독",\n    "영화 제공 가능": "청소년기의 상처와 성장, 사랑의 복잡함을 탐구하며, 자기 수용과 우정의 가치를 일깨움"\n  },\n  "기생충": {\n    "영화 타겟 취향": "사회적 메시지와 블랙 코미디 선호",\n    "영화 타겟 감정": "불편함, 씁쓸함, 공감",\n    "영화 제공 가능": "계급 간의 갈등을 통해 사회 구조를 비판하며, 관객에게 깊은 생각을 하게 함"\n  },\n  "메멘토": {\n    "영화 타겟 취향": "심리 스릴러, 복잡한 이야기 구조 선호",\n    "영화 타겟 감정": "혼란, 긴장감, 호기심",\n    "영화 제공 가능": "기억과 정체성에 대한 질문을 던지며, 관객을 몰입시키는 독특한 서사 구조"\n  },\n  "노인을 위한 나라는 없다": {\n    "영화 타겟 취향": "어두운 서사, 현대 사회의 부조리 선호",\n    "영화 타겟 감정": "불안, 긴장, 허무",\n    "영화 제공 가능": "폭력과 인간 본성에 대한 깊은 성찰을 제공하며, 도덕적 딜레마를 탐구"\n  },\n  "파이트 클럽": {\n    "영화 타겟 취향": "반문화적 메시지, 심리적 드라마 선호",\n    "영화 타겟 감정": "혼란, 반항, 카타르시스",\n    "영화 제공 가능": "소비 사회에 대한 비판과 개인 정체성 탐구, 강렬한 감정적 경험"\n  },\n  "박화영": {\n    "영화 타겟 취향": "사회적 이슈, 청소년 문제 선호",\n    "영화 타겟 감정": "불쾌감, 동정, 분노",\n    "영화 제공 가능": "가출 청소년의 현실을 적나라하게 보여주며, 심리적 고통과 인간 관계의 복잡성을 탐구"\n  },\n  "거인": {\n    "영화 타겟 취향": "독립 영화, 사회적 메시지 선호",\n    "영화 

In [30]:
review = review_json.replace('```json\n', '').replace('\n```', '').replace('\n', '').replace('\\n', '')
review = json.loads(review)
review

{'월플라워': {'영화 타겟 취향': '청춘, 성장, 감정적인 이야기 선호',
  '영화 타겟 감정': '공감, 감동, 질투, 고독',
  '영화 제공 가능': '청소년기의 상처와 성장, 사랑의 복잡함을 탐구하며, 자기 수용과 우정의 가치를 일깨움'},
 '기생충': {'영화 타겟 취향': '사회적 메시지와 블랙 코미디 선호',
  '영화 타겟 감정': '불편함, 씁쓸함, 공감',
  '영화 제공 가능': '계급 간의 갈등을 통해 사회 구조를 비판하며, 관객에게 깊은 생각을 하게 함'},
 '메멘토': {'영화 타겟 취향': '심리 스릴러, 복잡한 이야기 구조 선호',
  '영화 타겟 감정': '혼란, 긴장감, 호기심',
  '영화 제공 가능': '기억과 정체성에 대한 질문을 던지며, 관객을 몰입시키는 독특한 서사 구조'},
 '노인을 위한 나라는 없다': {'영화 타겟 취향': '어두운 서사, 현대 사회의 부조리 선호',
  '영화 타겟 감정': '불안, 긴장, 허무',
  '영화 제공 가능': '폭력과 인간 본성에 대한 깊은 성찰을 제공하며, 도덕적 딜레마를 탐구'},
 '파이트 클럽': {'영화 타겟 취향': '반문화적 메시지, 심리적 드라마 선호',
  '영화 타겟 감정': '혼란, 반항, 카타르시스',
  '영화 제공 가능': '소비 사회에 대한 비판과 개인 정체성 탐구, 강렬한 감정적 경험'},
 '박화영': {'영화 타겟 취향': '사회적 이슈, 청소년 문제 선호',
  '영화 타겟 감정': '불쾌감, 동정, 분노',
  '영화 제공 가능': '가출 청소년의 현실을 적나라하게 보여주며, 심리적 고통과 인간 관계의 복잡성을 탐구'},
 '거인': {'영화 타겟 취향': '독립 영화, 사회적 메시지 선호',
  '영화 타겟 감정': '우울, 공감, 고독',
  '영화 제공 가능': '사회적 고립과 개인의 내면적 갈등을 깊이 있게 다룸'},
 '끝까지 간다': {'영화 타겟 취향': '스릴러, 긴장감 있는 서사 선호',
  '영화 타겟 감정': '긴장

# 어떤 영화를 추천해주면 괜찮을지 제시

In [51]:
llm = ChatOpenAI( temperature=0.4, 
                 model='gpt-4o-mini')

recom_prompt = """ 너는 영화 추천 전문가야. 너의 역할은 사용자 입력을 받고 입력과 가장 유사한 영화를 리뷰 내역에서 추천해주면 되고 추천한 근거가 합리적이어야 해.
총 2개의 영화를 추천해줘. 추천 양식은 json 형태로 반환해. json 형태는 다음과 같아.
추천하는 영화 : 제목
추천하는 근거 : 사용자 입력과 영화 리뷰 내역 중 유사한 부분 중심으로 설명

사용자 입력 : {query}
영화 리뷰 : {review}
JSON object:
"""

recom_prompt_template = PromptTemplate(template=recom_prompt, input_variables=["query", "review"])

recom_chain = recom_prompt_template | llm | StrOutputParser()

In [52]:
# query = "산다는건 뭘까? 요즘 고민이 많이 돼. 왜 모두들 아등바등 살아가고 있는걸까? 조금이라도 쉬면 나만 뒤쳐지는 느낌이 들어."
# '```json\n{\n  "사용자의 취향": None,\n  "사용자의 감정적 상태": "고민, 불안",\n  "사용자에게 필요한 것": "휴식과 자기 성찰의 시간"\n}\n```'
query5 = result1_json
recom1_json = recom_chain.invoke({"query":query1, "review":review})
recom1_json

'```json\n{\n  "추천하는 영화": [\n    {\n      "제목": "거인",\n      "추천하는 근거": "사용자 입력에서 느끼는 고독과 삶의 고민은 \'거인\'의 주제와 밀접한 관련이 있습니다. 이 영화는 사회적 고립과 개인의 내면적 갈등을 깊이 있게 다루며, 관객에게 감정적으로 공감할 수 있는 요소를 제공합니다."\n    },\n    {\n      "제목": "월플라워",\n      "추천하는 근거": "사용자의 고민과 감정은 \'월플라워\'의 청춘과 성장 이야기와 유사합니다. 이 영화는 청소년기의 상처와 성장, 그리고 자기 수용의 가치를 탐구하며, 관객이 느끼는 고독과 감정의 복잡함에 대해 깊이 있는 통찰을 제공합니다."\n    }\n  ]\n}\n```'

In [53]:
# query = "난 요즘 너무 행복해. 인생 뭐 있어? 그냥 되는대로 사는거지! 오늘 하루 숨 쉬며 살아감에 감사해"
# '```json\n{\n  "사용자의 취향": None,\n  "사용자의 감정적 상태": "행복",\n  "사용자에게 필요한 것": "감사하는 마음을 더욱 키우고 즐거운 순간을 기록할 수 있는 일기나 다이어리를 추천합니다."\n}\n```'
query6 = result2_json
recom2_json = recom_chain.invoke({"query":query2, "review":review})
recom2_json

'```json\n{\n  "추천하는 영화": [\n    {\n      "제목": "포레스트 검프",\n      "추천하는 근거": "사용자가 인생에 감사하며 긍정적인 마음가짐을 표현한 것과 유사하게, \'포레스트 검프\'는 다양한 인생 경험을 통해 희망과 사랑의 메시지를 전달합니다. 영화의 주인공은 어려운 상황 속에서도 긍정적인 태도로 삶을 살아가며, 그 과정에서 많은 사람들에게 감동을 주는 이야기를 담고 있습니다."\n    },\n    {\n      "제목": "어바웃타임",\n      "추천하는 근거": "사용자가 하루하루 숨 쉬며 살아감에 감사하다는 표현과 잘 어울리는 영화로, \'어바웃타임\'은 사랑과 가족의 소중함을 일깨우며 시간의 가치에 대해 생각하게 합니다. 주인공은 시간을 되돌릴 수 있는 능력을 통해 소중한 순간들을 다시 경험하며, 삶의 작은 것들에 대한 감사함을 느끼게 되는 이야기입니다."\n    }\n  ]\n}\n```'

In [54]:
# query = "요즘 운동을 시작했어. 원래는 유산소를 좋아했는데 근육 펌핑 되는거 보니까 희열이 느껴져. 살아있는 기분 짜릿해"
# '```json\n{\n  "사용자의 취향": "유산소 운동",\n  "사용자의 감정적 상태": "희열, 짜릿함",\n  "사용자에게 필요한 것": "더 다양한 근력 운동 프로그램이나 동기 부여를 제공하는 커뮤니티"\n}\n```'
query7 = result3_json
recom3_json = recom_chain.invoke({"query":query3, "review":review})
recom3_json

'```json\n{\n  "추천하는 영화": [\n    {\n      "제목": "글로리 로드",\n      "추천하는 근거": "사용자가 운동을 시작하고 근육 펌핑의 희열을 느끼고 있다는 점에서, \'글로리 로드\'는 스포츠를 통한 인종 차별 문제와 팀워크의 중요성을 다루며 감동적인 이야기를 전달합니다. 운동의 열정과 감동을 느낄 수 있는 영화입니다."\n    },\n    {\n      "제목": "머니볼",\n      "추천하는 근거": "사용자가 운동을 통해 느끼는 희열과 도전의 중요성에 공감할 수 있는 영화로, \'머니볼\'은 혁신과 도전의 중요성을 강조하며 스포츠를 통해 인생의 교훈을 전달합니다. 이는 사용자의 새로운 운동 시작과 잘 어울립니다."\n    }\n  ]\n}\n```'

In [55]:
# query = "요즘 운동을 시작했어. 원래는 유산소를 좋아했는데 근육 펌핑 되는거 보니까 희열이 느껴져. 살아있는 기분 짜릿해"
# '```json\n{\n  "사용자의 취향": "반짝이는 것, 강렬한 경험",\n  "사용자의 감정적 상태": "놀람, 짜릿함",\n  "사용자에게 필요한 것": "안정감과 편안함을 줄 수 있는 환경"\n}\n```'
query8 = result4_json
recom4_json = recom_chain.invoke({"query":query4, "review":review})
recom4_json

'```json\n{\n  "추천하는 영화": [\n    {\n      "제목": "헤드윅",\n      "추천하는 근거": "사용자 입력의 \'반짝반짝\'과 \'젖은 눈빛\'이라는 표현은 감정의 깊이와 미적 요소를 강조합니다. \'헤드윅\'은 자아 찾기와 사랑에 대한 이야기를 통해 관객에게 깊은 감정적 여운을 남기며, 이러한 감정의 복잡함을 잘 표현하고 있습니다."\n    },\n    {\n      "제목": "어바웃타임",\n      "추천하는 근거": "사용자 입력에서 느껴지는 짜릿함과 감정의 흐름은 \'어바웃타임\'의 따뜻한 가족 이야기와 잘 어울립니다. 이 영화는 사랑과 가족의 소중함을 일깨우며, 시간의 가치에 대해 생각하게 하는 감동적인 요소가 있습니다."\n    }\n  ]\n}\n```'

# 사용자 입력 취향 추출 자동화..?

In [56]:
llm = ChatOpenAI( temperature=0.4, 
                 model='gpt-4o-mini')

prompt = """ 너는 사용자의 얘기를 듣고 사용자의 언어간 유사성을 통해 분류해.
json 형태로 key 에는 분류 대표 키워드를 작성하고 항목으론 내용을 작성하도록 해. json 형태는 다음과 같아.
분류 키워드1 : 내용
분류 키워드2 : 내용
분류 키워드는 더 많아지는 경우 더 추가하면 돼.
사용자의 응답 : {query}
JSON object : 
"""


prompt_template = PromptTemplate(template=prompt, input_variables=["comments"])

recom_chain = prompt_template | llm | StrOutputParser()

In [57]:
query = "내일 모든 일이 끝나고 집에 오면 냉장고에 있는 레몬 하이볼을 꺼내서 얼음컵에 넣고 한잔 딱 마실거야. 너무 행복하겠다."
result_json = recom_chain.invoke({"query":query})
result_json

'```json\n{\n    "행복한 순간": "내일 모든 일이 끝나고 집에 오면 냉장고에 있는 레몬 하이볼을 꺼내서 얼음컵에 넣고 한잔 딱 마실거야. 너무 행복하겠다.",\n    "음료": "레몬 하이볼",\n    "일상": "모든 일이 끝나고 집에 오는 것"\n}\n```'